# **Set Up (Google Collab NB)**

In [ ]:
# Install the required package for generating embeddings
!pip install -U sentence-transformers

# Standard library imports
import json

# Third-party imports for math and ML models
import numpy as np
from sentence_transformers import SentenceTransformer

print("Setup complete! Libraries imported and ready to go.")

Setup complete! Libraries imported and ready to go.


# **Question 1**

In [ ]:
import pandas as pd

file_path = '/content/tweets-utf-8.json'
df = pd.read_json(file_path, lines=True)

display(df.head())

,_id,created_at,id,id_str,text,display_text_range,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,...,lang,timestamp_ms,extended_entities,possibly_sensitive,quoted_status_id,quoted_status_id_str,quoted_status,extended_tweet,retweeted_status,withheld_in_countries
0,{'$oid': '5aa588b2c4214f42f4c622dd'},2017-04-17 02:49:03+00:00,{'$numberLong': '853802458841317377'},853802458841317376,@allisonsimss Love you lots❤,"[14, 28]","<a href=""http://twitter.com/download/iphone"" r...",False,{'$numberLong': '853802287550144513'},8.538023e+17,...,en,2017-04-17 02:49:03.727,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,{'$oid': '5aa588b2c4214f42f4c622de'},2017-04-17 02:49:12+00:00,{'$numberLong': '853802493968613376'},853802493968613376,Can't wait to read the newspaper tmrw😂,NaN,"<a href=""http://twitter.com/download/iphone"" r...",False,None,NaN,...,en,2017-04-17 02:49:12.102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,{'$oid': '5aa588b2c4214f42f4c622df'},2017-04-17 02:49:33+00:00,{'$numberLong': '853802585433792514'},853802585433792512,happy easter ☺️🐰💖 @AlexZschering https://t.co/...,"[0, 32]","<a href=""http://twitter.com/download/iphone"" r...",False,None,NaN,...,en,2017-04-17 02:49:33.909,{'media': [{'id': {'$numberLong': '85380256744...,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,{'$oid': '5aa588b2c4214f42f4c622e0'},2017-04-17 02:49:41+00:00,{'$numberLong': '853802619193761793'},853802619193761792,@Nancy_Cholette Lâche pas. 😆,"[16, 28]","<a href=""http://twitter.com/download/android"" ...",False,{'$numberLong': '853802403753328644'},8.538024e+17,...,fr,2017-04-17 02:49:41.958,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,{'$oid': '5aa588b2c4214f42f4c622e1'},2017-04-17 02:49:42+00:00,{'$numberLong': '853802621777432576'},853802621777432576,Even if your not a #Survivor fan this intervie...,"[0, 88]","<a href=""http://twitter.com/download/iphone"" r...",False,None,NaN,...,en,2017-04-17 02:49:42.574,NaN,0.0,{'$numberLong': '853501208266510336'},8.535012e+17,{'created_at': 'Sun Apr 16 06:51:59 +0000 2017...,NaN,NaN,NaN


In [ ]:
def get_tweets(filepath="tweets-utf-8.json"):
    """
    Reads the JSON file containing tweets and extracts the text of each tweet.
    The file is formatted with one JSON object per line, so we iterate line by line.
    """
    tweet_texts = []

    # Using a context manager (with open) is best practice so it automatically closes the file
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            # Parse the JSON string from the current line into a Python dictionary
            tweet_data = json.loads(line.strip())

            # Extract the 'text' field as requested and append it to our list
            if 'text' in tweet_data:
                tweet_texts.append(tweet_data['text'])

    return tweet_texts

# Check to make sure it's reading everything correctly
tweets = get_tweets()
print(f"Successfully loaded {len(tweets)} tweets!")
print(f"Sample tweet: {tweets[0]}")

Successfully loaded 110474 tweets!
Sample tweet: @allisonsimss Love you lots❤


# **Question 2**

In [ ]:
def sort_by_sim(query_embedding, document_embeddings, documents):
    """
    Computes cosine similarity between a query embedding and a list of doc embeddings.
    Returns a list of (similarity, document) tuples sorted in descending order.
    """
    results = []

    # Calculate the L2 norm (magnitude) of the query vector once to save compute
    query_norm = np.linalg.norm(query_embedding)

    # Iterate through both the embeddings and the raw text simultaneously
    for doc_emb, doc_text in zip(document_embeddings, documents):
        doc_norm = np.linalg.norm(doc_emb)

        # Edge case check: Avoid division by zero if we get a blank/zero vector
        if query_norm == 0 or doc_norm == 0:
            sim = 0.0
        else:
            # Cosine similarity formula: dot product divided by the product of the norms
            sim = np.dot(query_embedding, doc_emb) / (query_norm * doc_norm)

        results.append((sim, doc_text))

    # Sort the list of tuples by the first element (similarity score) in descending order
    # reverse=True ensures the highest similarity scores are at the top
    results.sort(key=lambda x: x[0], reverse=True)

    return results

# Optional sanity check to make sure the math works with some dummy data
dummy_query = np.array([1, 0, 1])
dummy_docs_emb = [np.array([1, 0, 1]), np.array([0, 1, 0]), np.array([0.5, 0, 0.5])]
dummy_docs = ["Identical vector", "Orthogonal vector", "Same direction, smaller magnitude"]

print("Sanity Check for sort_by_sim:")
for score, doc in sort_by_sim(dummy_query, dummy_docs_emb, dummy_docs):
    print(f"Score: {score:.4f} | Doc: {doc}")

Sanity Check for sort_by_sim:
Score: 1.0000 | Doc: Identical vector
Score: 1.0000 | Doc: Same direction, smaller magnitude
Score: 0.0000 | Doc: Orthogonal vector


# **Question 3**

In [ ]:
def top25_glove(tweets_list):
    """
    Loads the GloVe model, encodes the query and the tweets,
    and returns the top 25 most similar tweets using our sort_by_sim function.
    """
    # 1. Load the pre-trained GloVe sentence embedding model specified by the prof
    print("Loading GloVe model... (this might take a few seconds the first time)")
    glove_model = SentenceTransformer('average_word_embeddings_glove.840B.300d')

    # 2. Define our target query
    query = "I am looking for a job."

    # 3. Encode the query and the entire list of tweets into vector embeddings
    # SentenceTransformer outputs numpy arrays by default, which is perfect for our sort_by_sim function
    print("Encoding query and tweets...")
    query_embedding = glove_model.encode(query)
    tweet_embeddings = glove_model.encode(tweets_list)

    # 4. Calculate similarities and sort them using our previously built function
    print("Calculating similarities...")
    sorted_tweets = sort_by_sim(query_embedding, tweet_embeddings, tweets_list)

    # 5. Return only the top 25 results
    return sorted_tweets[:25]

# Let's run it and print the results to make sure it looks reasonable
glove_results = top25_glove(tweets)

print("\n--- Top 25 Tweets (GloVe) ---")
for rank, (score, tweet_text) in enumerate(glove_results, 1):
    print(f"{rank}. [Score: {score:.4f}] {tweet_text}")

Loading GloVe model... (this might take a few seconds the first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

wordembedding_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding query and tweets...
Calculating similarities...

--- Top 25 Tweets (GloVe) ---
1. [Score: 0.8553] I love my job. 😂😐😭
2. [Score: 0.7630] @SusanSherring You guys are doing a great job.
3. [Score: 0.7313] @realDonaldTrump my grandpa wanted to say good job. So good job. From Phil
4. [Score: 0.7055] @JoannaLidback @ChittendenNate @cabotcheese Good job. Other than you look like you wish you were somewhere else...
5. [Score: 0.6777] Getting ready for another day on the job. @tutordoctor https://t.co/nlDTfKUN8J
6. [Score: 0.6698] Yes I did! I liked it a lot and looking forward to more!! https://t.co/foQccgQjFX
7. [Score: 0.6568] @carbonfixated I can't imagine anyone having the patience for it but I bet she'd do an excellent job.
8. [Score: 0.6491] @EdBrown19 @ThatFishCreigh @Fffeisty Wow Ed, nice job.
9. [Score: 0.6480] @anjacks0n Perish the thought, seriously! Working on this stuff is the favourite part of my job. :)
10. [Score: 0.6388] How I feel looking at all the work I have to do

# **Question 4**

In [ ]:
def top25_minilm(tweets_list):
    """
    Loads the MiniLM model (a BERT derivative), encodes the query and the tweets,
    and returns the top 25 most similar tweets using our sort_by_sim function.
    """
    # 1. Load the pre-trained MiniLM sentence embedding model
    print("Loading all-MiniLM-L6-v2 model...")
    minilm_model = SentenceTransformer('all-MiniLM-L6-v2')

    # 2. Define our target query
    query = "I am looking for a job."

    # 3. Encode the query and the entire list of tweets into vector embeddings
    # The prof warned this takes ~10 mins locally, but Colab handles it much faster!
    print("Encoding query and tweets (hang tight, this is a beefier model)...")
    query_embedding = minilm_model.encode(query)
    tweet_embeddings = minilm_model.encode(tweets_list)

    # 4. Calculate similarities and sort them using our previously built function
    print("Calculating similarities...")
    sorted_tweets = sort_by_sim(query_embedding, tweet_embeddings, tweets_list)

    # 5. Return only the top 25 results
    return sorted_tweets[:25]

# Let's run it and print the results to see how it compares to GloVe
minilm_results = top25_minilm(tweets)

print("\n--- Top 25 Tweets (MiniLM) ---")
for rank, (score, tweet_text) in enumerate(minilm_results, 1):
    print(f"{rank}. [Score: {score:.4f}] {tweet_text}")

Loading all-MiniLM-L6-v2 model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding query and tweets (hang tight, this is a beefier model)...
Calculating similarities...

--- Top 25 Tweets (MiniLM) ---
1. [Score: 0.8160] I need a job
2. [Score: 0.6099] Does anyone know of anywhere that's hiring 🆘🙂
3. [Score: 0.5766] Can you recommend anyone for this #job? Retail Clerk (Part-Time) - 6001 Highland Road, Whie Lake, MI 48836 - https://t.co/aZkkb2v11x
4. [Score: 0.5636] If you're looking for work in #Shelby, MI, check out this #job: https://t.co/0wP0eSzAS7 #cfgjobs #Hiring
5. [Score: 0.5601] If you're looking for work in #Shelby, MI, check out this #job: https://t.co/SRBQ9IPq97 #Retail #Hiring
6. [Score: 0.5599] If you're looking for work in #Shelby, MI, check out this #job: https://t.co/Js7LQEpgxX #Hiring
7. [Score: 0.5539] Can you recommend anyone for this #job? Receiving Clerk - 10 Mile Rd NE, Rockford MI - https://t.co/8ozG1BTp5d #SupplyChain #Rockford, MI
8. [Score: 0.5424] If you're looking for work in #UNION, MI, check out this #job: https://t.co/PfYe1OaG20